In [4]:
import os 

In [5]:
%pwd

'c:\\Users\\Asus\\Machine_learning\\LLM\\Language_Model\\GPT2_124M\\notebook'

In [6]:
os.chdir("..//")

In [7]:
os.chdir("..//")

In [ ]:
import time 
import math 
import torch
import torch.nn as nn
from torch.nn import functional as F
from dataclasses import dataclass

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [7]:
class CasualSelfAttention(nn.Module):
    def __init__(self,config):
        super().__init__()
        assert config.n_emb % config.n_head ==0
        self.c_attn = nn.Linear(config.n_emb,3*config.n_emb)    # combined attention==> key,query,value projection for all heads,but in a batch
        self.c_proj = nn.Linear(config.n_emb,config.n_emb)      # output projection

        self.n_head = config.n_head
        self.n_emb  = config.n_emb
        self.register_buffer("bias",torch.tril(torch.ones(config.block_size,config.block_size)).view(1,1,config.block_size,config.block_size))

    def forward(self,x):
        B,T,C       = x.shape # batch_size, sequence length, embedding dim
        # calculate query, key, values for all heads in batch and move head forward to be the batch dim
        # nh                    = "number of heads",
        # hs                    = "head size"
        # C (number of channels)= nh * hs
        # e.g. in GPT-2 (124M)==> n_head    =12,
        #                         hs        =64, ==> nh * hs = C = 768 channels in the Transformer
        qkv     = self.c_attn(x)                                                    # B,T,3*n_emb
        q,k,v   = qkv.split(self.n_emb,dim=2)                                       # B,T,n_emb
        q       = q.view(B,T,self.n_head,self.n_emb//self.n_head).transpose(1,2)    # B, sequence_length(T), n_heads(n_h), head_size(hs) ==> B, n_h,T,hs
        k       = k.view(B,T,self.n_head,self.n_emb//self.n_head).transpose(1,2)    # B, sequence_length(T), n_heads(n_h), head_size(hs) ==> B, n_h,T,hs
        v       = v.view(B,T,self.n_head,self.n_emb//self.n_head).transpose(1,2)    # B, sequence_length(T), n_heads(n_h), head_size(hs) ==> B, n_h,T,hs
        ## Attention 
        atten   = (q@k.transpose(-2,-1)) * (1.0/math.sqrt(k.size(-1)))
        atten   = atten.masked_fill(self.bias[:,:,:T,:T] == 0,float("-inf"))
        atten   = F.softmax(atten,dim=-1)

        y       = atten @ v                 # (B,nh,T,T) x (B,nh,T,hs) ==> (B,nh,T,hs)
        y       = y.transpose(1,2).contiguous().view(B,T,C)
        #output projection
        y       = self.c_proj(y)
        return y

class MLP(nn.Module):
    def __init__(self,config):
        super().__init__()
        self.c_fc   = nn.Linear(config.n_emb,4*config.n_emb)
        self.gelu   = nn.GELU(approximate="tanh")   # there is no reason to use this approximation in nowdays, the time they develop this approximation they faced speed issue. thats why developed approximation
        self.c_proj = nn.Linear(config.n_emb * 4,config.n_emb)
    def forward(self,x):
        x   = self.c_fc(x)
        x   = self.gelu(x)
        x   = self.c_proj(x)
        return x
class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1   = nn.LayerNorm(config.n_emb)
        self.attn   = CasualSelfAttention(config)
        self.ln_2   = nn.LayerNorm(config.n_emb)
        self.mlp    = MLP(config)

    def forward(self,x):
        x   = x + self.attn(self.ln_1(x))
        x   = x + self.mlp(self.ln_2(x))
        return x


In [8]:
@dataclass
class GPTConfig:
    block_size:int  = 1024  # ==> block size
    vocab_size:int  = 50257 # ==> numbers of tokens ==> 50000 merges + 256 bytes token + 1 special token <|endoftext|>
    n_layer:int     = 12
    n_head:int      = 12
    n_emb:int       = 768   # ==> embedding dim

In [9]:
import tiktoken

In [10]:
class DataLoaderLite:
    def __init__(self,B,T):
        self.B  = B     # batch 
        self.T  = T     # sequence length 
        with open("data\gpt_train.txt","r") as f:
            text    = f.read()
        enc         = tiktoken.get_encoding("gpt2")
        tokens      = enc.encode(text)
        self.tokens = torch.tensor(tokens)
        print(f"loaded of {len(self.tokens)} tokens")
        print(f"1 Epoch = {len(self.tokens) // (B*T)} Batches of token")

        self.current_position = 0 

    def next_batch(self):
        B,T     = self.B,self.T
        buff    = self.tokens[self.current_position:self.current_position+B*T+1]
        x       = (buff[:-1]).view(B,T)     # input 
        y       = (buff[1:]).view(B,T)      # target 
        self.current_position   += B*T 
        if self.current_position + (B*T+1)> len(self.tokens):
            self.current_position = 0 
        return x,y

In [11]:
class GPT(nn.Module):
  def __init__(self,config):
      super().__init__()
      self.config = config
      self.transformer = nn.ModuleDict(dict(
          wte     = nn.Embedding(config.vocab_size,config.n_emb),           # token embeding
          wpe     = nn.Embedding(config.block_size,config.n_emb),           # position embedding
          h       = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),    # self attention heads
          ln_f    = nn.LayerNorm(config.n_emb)
      ))
      self.lm_head= nn.Linear(config.n_emb,config.vocab_size,bias=False)    # lm_head is following be softmax, and bias not make any sence or improvement in learning.
      # The bias term in this case would just add a constant to each token’s logit — this doesn’t meaningfully improve learning,
      #-----------------------weight sharing scheme ---------------------------------# 
      self.transformer.wte.weight  = self.lm_head.weight
      # ----------------------Parameter Initialization ------------------------------#
      
  
  def forward(self,idx,target=None):
    # shape of idx is (B,T)
    B,T     = idx.shape
    assert T<=self.config.block_size, f"cannot forward sequence of length {T},block_size is only {self.config.block_size}"
    pos     = torch.arange(0,T,dtype=torch.long,device=idx.device)  # shape (T)
    pos_emb = self.transformer.wpe(pos)                             # position embedding of shape (_,T,n_emb)
    tok_emb = self.transformer.wte(idx)                             # token embedding of shape    (B,T,n_emb)

    x       = tok_emb + pos_emb         # (B,T,n_emb)
    for block in self.transformer.h:
      x = block(x)
    #forward the final layerorm and classifier
    x       = self.transformer.ln_f(x)
    logits  = self.lm_head(x)           # (B,T,n_emb)

    ##------------------------------Adding Target and Loss---------------------- ##
    loss    = None
    if target is None:
      loss  = None
    elif target is not None:
      loss  = F.cross_entropy(input   = logits.view(-1,logits.size(-1)),        # cross entropy does not like multi-dimensional input, flatten out into 2D
                              target  = target.view(-1),)
    return logits,loss

In [12]:

train_loader = DataLoaderLite(B=4,T=32)

loaded of 338025 tokens
1 Epoch = 2640 Batches of token


In [13]:
model = GPT(GPTConfig())
model.eval()
model = model.to(device)

In [14]:
## Optimizer 
optimizer = torch.optim.AdamW(model.parameters(),lr=3e-4)
for i in range(50):
    x,y = train_loader.next_batch()
    x,y = x.to(device),y.to(device)
    t0  = time.time()
    optimizer.zero_grad()
    logits,loss = model(x,y)
    loss.backward()
    optimizer.step()
    t1  = time.time()
    dt = (t1 - t0) * 1000  # convert seconds to milliseconds
    print(f"Step : {i} loss: {loss.item():.4f}, dt: {dt:.4f} ms")

c:\Users\Asus\anaconda3\envs\llm_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Step : 0 loss: 11.0170, dt: 246.4700 ms
Step : 1 loss: 9.8597, dt: 34.5039 ms
Step : 2 loss: 9.1922, dt: 30.4899 ms
Step : 3 loss: 9.1908, dt: 39.8157 ms
Step : 4 loss: 8.6999, dt: 28.0344 ms
Step : 5 loss: 8.3570, dt: 22.5065 ms
Step : 6 loss: 8.8369, dt: 35.0177 ms
Step : 7 loss: 8.6952, dt: 32.6326 ms
Step : 8 loss: 8.1225, dt: 28.8985 ms
Step : 9 loss: 7.8843, dt: 38.5249 ms
Step : 10 loss: 8.3472, dt: 30.6892 ms
Step : 11 loss: 7.3403, dt: 29.6891 ms
Step : 12 loss: 7.8279, dt: 35.5787 ms
Step : 13 loss: 7.3942, dt: 29.8839 ms
Step : 14 loss: 7.4439, dt: 33.4032 ms
Step : 15 loss: 7.3540, dt: 44.6997 ms
Step : 16 loss: 7.3850, dt: 58.7068 ms
Step : 17 loss: 8.1940, dt: 39.8030 ms
Step : 18 loss: 7.2683, dt: 30.9126 ms
Step : 19 loss: 7.7883, dt: 26.4604 ms
Step : 20 loss: 7.5526, dt: 20.9818 ms
Step : 21 loss: 7.8247, dt: 42.7089 ms
Step : 22 loss: 6.5094, dt: 27.8237 ms
Step : 23 loss: 6.8492, dt: 22.9082 ms
Step : 24 loss: 6.8068, dt: 30.4923 ms
Step : 25 loss: 6.7135, dt: 26.48

## weight Initialization

In [15]:
class GPT(nn.Module):
  def __init__(self,config):
      super().__init__()
      self.config = config
      self.transformer = nn.ModuleDict(dict(
          wte     = nn.Embedding(config.vocab_size,config.n_emb),           # token embeding
          wpe     = nn.Embedding(config.block_size,config.n_emb),           # position embedding
          h       = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),    # self attention heads
          ln_f    = nn.LayerNorm(config.n_emb)
      ))
      self.lm_head= nn.Linear(config.n_emb,config.vocab_size,bias=False)    # lm_head is following be softmax, and bias not make any sence or improvement in learning.
      # The bias term in this case would just add a constant to each token’s logit — this doesn’t meaningfully improve learning,
      #-----------------------weight sharing scheme ---------------------------------# 
      self.transformer.wte.weight  = self.lm_head.weight
      # ----------------------Parameter Initialization ------------------------------#
      self.apply(self._init_weights) 

  def _init_weights(self,module):
    if isinstance(module,nn.Linear):
      torch.nn.init.normal_(module.weight,mean=0.0,std=0.02)  # weight initialization function 
      if module.bias is not None:
        torch.nn.init.zeros_(module.bias) 
    elif isinstance(module,nn.Embedding):
      torch.nn.init.normal_(module.weight,mean=0.0,std=0.02) 

  def forward(self,idx,target=None):
    # shape of idx is (B,T)
    B,T     = idx.shape
    assert T<=self.config.block_size, f"cannot forward sequence of length {T},block_size is only {self.config.block_size}"
    pos     = torch.arange(0,T,dtype=torch.long,device=idx.device)  # shape (T)
    pos_emb = self.transformer.wpe(pos)                             # position embedding of shape (_,T,n_emb)
    tok_emb = self.transformer.wte(idx)                             # token embedding of shape    (B,T,n_emb)

    x       = tok_emb + pos_emb         # (B,T,n_emb)
    for block in self.transformer.h:
      x = block(x)
    #forward the final layerorm and classifier
    x       = self.transformer.ln_f(x)
    logits  = self.lm_head(x)           # (B,T,n_emb)

    ##------------------------------Adding Target and Loss---------------------- ##
    loss    = None
    if target is None:
      loss  = None
    elif target is not None:
      loss  = F.cross_entropy(input   = logits.view(-1,logits.size(-1)),        # cross entropy does not like multi-dimensional input, flatten out into 2D
                              target  = target.view(-1),)
    return logits,loss

- Initialized `token embedding` std = 0.02    (documented in GPT2 code that released by OpenAi)
- Initialized `position embedding` std = 0.01 (documented in GPT2 code that released by OpenAi), But we does not change to 0.01 for above bias.    

- Initialized `Bias` with 0.  

- Other layer required to initialize is `LayerNorm` layers. 
- By following xavier 


## NOTE 1: 

- NEVER initialize all weights to zero or any other constant value (unless it's for biases, which are often initialized to zero).

- **Reason**: 
    - If all weights in a layer are the same, every neuron in that layer will produce the same output and receive the same gradient update during backpropagation. - This means all neurons in a layer will learn the exact same features, making the network redundant and significantly limiting its learning capacity. This is known as the symmetry problem.

# NOTE 2:

- standard deviation grow inside the residual connection, let me give u sample 
```python
import torch

n_emb = 768
n = 12

x = torch.zeros(n_emb)

for i in range(n):
    x = x + torch.randn(n_emb)

print("Standard deviation after residual additions:", x.std())
- The x.std()  value would be really big

In [ ]:
n_emb = 768 
n = 100 # number of residual layers

x = torch.zeros(n_emb) 
for i in range(n):
    x += n ** -0.5 * torch.randn(n_emb) # ==> n ** (-0.5) ==>>  1/ sqrt(n)


print(x.std())

tensor(0.9872)


In [17]:
#### Solve the issue with multipy it with 1 / sqrt(number of residual layers in block)

In [18]:
class CasualSelfAttention(nn.Module):
    def __init__(self,config):
        super().__init__()
        assert config.n_emb % config.n_head ==0
        self.c_attn = nn.Linear(config.n_emb,3*config.n_emb)    # combined attention==> key,query,value projection for all heads,but in a batch
        self.c_proj = nn.Linear(config.n_emb,config.n_emb)      # output projection
        self.c_proj.NANOGPT_SCALE_INIT = 1 # flaging this instance -->> Intitialization method <<--  

        self.n_head = config.n_head
        self.n_emb  = config.n_emb
        self.register_buffer("bias",torch.tril(torch.ones(config.block_size,config.block_size)).view(1,1,config.block_size,config.block_size))

    def forward(self,x):
        B,T,C       = x.shape # batch_size, sequence length, embedding dim
        # calculate query, key, values for all heads in batch and move head forward to be the batch dim
        # nh                    = "number of heads",
        # hs                    = "head size"
        # C (number of channels)= nh * hs
        # e.g. in GPT-2 (124M)==> n_head    =12,
        #                         hs        =64, ==> nh * hs = C = 768 channels in the Transformer
        qkv     = self.c_attn(x)                                                    # B,T,3*n_emb
        q,k,v   = qkv.split(self.n_emb,dim=2)                                       # B,T,n_emb
        q       = q.view(B,T,self.n_head,self.n_emb//self.n_head).transpose(1,2)    # B, sequence_length(T), n_heads(n_h), head_size(hs) ==> B, n_h,T,hs
        k       = k.view(B,T,self.n_head,self.n_emb//self.n_head).transpose(1,2)    # B, sequence_length(T), n_heads(n_h), head_size(hs) ==> B, n_h,T,hs
        v       = v.view(B,T,self.n_head,self.n_emb//self.n_head).transpose(1,2)    # B, sequence_length(T), n_heads(n_h), head_size(hs) ==> B, n_h,T,hs
        ## Attention 
        atten   = (q@k.transpose(-2,-1)) * (1.0/math.sqrt(k.size(-1)))
        atten   = atten.masked_fill(self.bias[:,:,:T,:T] == 0,float("-inf"))
        atten   = F.softmax(atten,dim=-1)

        y       = atten @ v                 # (B,nh,T,T) x (B,nh,T,hs) ==> (B,nh,T,hs)
        y       = y.transpose(1,2).contiguous().view(B,T,C)
        #output projection
        y       = self.c_proj(y)
        return y

class MLP(nn.Module):
    def __init__(self,config):
        super().__init__()
        self.c_fc   = nn.Linear(config.n_emb,4*config.n_emb)
        self.gelu   = nn.GELU(approximate="tanh")   # there is no reason to use this approximation in nowdays, the time they develop this approximation they faced speed issue. thats why developed approximation
        self.c_proj = nn.Linear(config.n_emb * 4,config.n_emb)
        self.c_proj.NANOGPT_SCALE_INIT = 1

    def forward(self,x):
        x   = self.c_fc(x)
        x   = self.gelu(x)
        x   = self.c_proj(x)
        return x
    
class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1   = nn.LayerNorm(config.n_emb)
        self.attn   = CasualSelfAttention(config)
        self.ln_2   = nn.LayerNorm(config.n_emb)
        self.mlp    = MLP(config)

    def forward(self,x):
        x   = x + self.attn(self.ln_1(x))
        x   = x + self.mlp(self.ln_2(x))
        return x


In [ ]:
class GPT(nn.Module):
  
  def __init__(self,config):
      super().__init__()
      self.config = config
      self.transformer = nn.ModuleDict(dict(
          wte     = nn.Embedding(config.vocab_size,config.n_emb),           # token embeding
          wpe     = nn.Embedding(config.block_size,config.n_emb),           # position embedding
          h       = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),    # self attention heads
          ln_f    = nn.LayerNorm(config.n_emb)
      ))
      self.lm_head= nn.Linear(config.n_emb,config.vocab_size,bias=False)    # lm_head is following be softmax, and bias not make any sence or improvement in learning.
      # The bias term in this case would just add a constant to each token’s logit — this doesn’t meaningfully improve learning,
      #-----------------------weight sharing scheme ---------------------------------# 
      self.transformer.wte.weight  = self.lm_head.weight
      # ----------------------Parameter Initialization ------------------------------#
      self.apply(self._init_weights) 

  def _init_weights(self,module):
    std = 0.02 
    if isinstance(module,nn.Linear):
      if hasattr(module,"NANOGPT_SCALE_INIT"):
        std *= (2* self.config.n_layer) ** -0.5  # In a block there is 2 residual connection per layer, so total n_layer * 2 residual connection total 
      # (2* self.config.n_layer) ** -0.5 ==> this means 1 / sqrt(total residual connection)
      torch.nn.init.normal_(module.weight,mean = 0.0,std = std)  # weight initialization function 
      if module.bias is not None:
        torch.nn.init.zeros_(module.bias) 
    elif isinstance(module,nn.Embedding):
      torch.nn.init.normal_(module.weight,mean=0.0,std = std) 

  def forward(self,idx,target=None):
    # shape of idx is (B,T)
    B,T     = idx.shape
    assert T<=self.config.block_size, f"cannot forward sequence of length {T},block_size is only {self.config.block_size}"
    pos     = torch.arange(0,T,dtype=torch.long,device=idx.device)  # shape (T)
    pos_emb = self.transformer.wpe(pos)                             # position embedding of shape (_,T,n_emb)
    tok_emb = self.transformer.wte(idx)                             # token embedding of shape    (B,T,n_emb)

    x       = tok_emb + pos_emb         # (B,T,n_emb)
    for block in self.transformer.h:
      x = block(x)
    #forward the final layerorm and classifier
    x       = self.transformer.ln_f(x)
    logits  = self.lm_head(x)           # (B,T,n_emb)

    ##------------------------------Adding Target and Loss---------------------- ##
    loss    = None
    if target is None:
      loss  = None
    elif target is not None:
      loss  = F.cross_entropy(input   = logits.view(-1,logits.size(-1)),        # cross entropy does not like multi-dimensional input, flatten out into 2D
                              target  = target.view(-1),)
    return logits,loss

In [20]:
torch.manual_seed(1337)
if torch.cuda.is_available():
    torch.cuda.manual_seed(1337)

train_loader = DataLoaderLite(B=4,T=32)

loaded of 338025 tokens
1 Epoch = 2640 Batches of token


In [21]:
model = GPT(GPTConfig())
model.eval()
model = model.to(device)

In [22]:
## Optimizer 
optimizer = torch.optim.AdamW(model.parameters(),lr=3e-4)
for i in range(50):
    x,y = train_loader.next_batch()
    x,y = x.to(device),y.to(device)
    t0  = time.time()
    optimizer.zero_grad()
    logits,loss = model(x,y)
    loss.backward()
    optimizer.step()
    t1  = time.time()
    dt = (t1 - t0) * 1000  # convert seconds to milliseconds
    print(f"Step : {i} loss: {loss.item():.4f}, dt: {dt:.4f} ms")

Step : 0 loss: 10.9600, dt: 47.5032 ms
Step : 1 loss: 9.6877, dt: 29.9442 ms
Step : 2 loss: 9.0829, dt: 35.3551 ms
Step : 3 loss: 9.1460, dt: 30.2086 ms
Step : 4 loss: 8.6262, dt: 35.4302 ms
Step : 5 loss: 8.3317, dt: 33.6196 ms
Step : 6 loss: 8.8980, dt: 29.2518 ms
Step : 7 loss: 8.8380, dt: 43.2124 ms
Step : 8 loss: 8.1160, dt: 51.2660 ms
Step : 9 loss: 8.0422, dt: 48.7826 ms
Step : 10 loss: 8.3808, dt: 29.9253 ms
Step : 11 loss: 7.4356, dt: 36.3920 ms
Step : 12 loss: 7.8246, dt: 28.1601 ms
Step : 13 loss: 7.4589, dt: 39.8245 ms
Step : 14 loss: 7.5319, dt: 30.7324 ms
Step : 15 loss: 7.3667, dt: 33.4623 ms
Step : 16 loss: 7.4368, dt: 33.5734 ms
Step : 17 loss: 8.2936, dt: 29.6633 ms
Step : 18 loss: 7.2028, dt: 45.4702 ms
Step : 19 loss: 7.8870, dt: 37.9734 ms
Step : 20 loss: 7.5059, dt: 35.1336 ms
Step : 21 loss: 7.8229, dt: 43.3183 ms
Step : 22 loss: 6.4254, dt: 28.6546 ms
Step : 23 loss: 6.8778, dt: 33.7787 ms
Step : 24 loss: 6.8273, dt: 28.9049 ms
Step : 25 loss: 6.7019, dt: 27.422